# Retinal Vessel Segmentation — Baseline vs TTT Evaluation
**MSc AI Computer Vision Project**

This notebook evaluates the trained U-Net model on three out-of-domain datasets (STARE, CHASE, HRF) with and without Test-Time Adaptation (TTT).

### Setup instructions
1. Go to **[colab.research.google.com](https://colab.research.google.com)** → File → Upload notebook → select this file
2. Runtime → Change runtime type → **T4 GPU** → Save
3. Run **Cell 1** — a file picker will appear → select **`colab_package.zip`** from your PC
4. Run all remaining cells in order

In [ ]:
# ── Cell 1: Upload colab_package.zip and extract ─────────────────────────────
from google.colab import files
import zipfile, os, sys

print("Select colab_package.zip when the file picker opens...")
uploaded = files.upload()   # opens file picker — select colab_package.zip

zip_name = list(uploaded.keys())[0]
print(f"Extracting {zip_name} (with path fix)...")
with zipfile.ZipFile(zip_name, 'r') as z:
    for member in z.infolist():
        member.filename = member.filename.replace('\\', '/')  # fix Windows paths
        z.extract(member, '/content/')

PROJECT_PATH = '/content/colab_package'
assert os.path.isdir(PROJECT_PATH), f"Extraction failed. Contents: {os.listdir('/content/')}"
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print("Ready. Working directory:", os.getcwd())

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q torch torchvision scikit-learn Pillow tqdm matplotlib

In [ ]:
# ── Cell 3: Verify GPU and imports ───────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found — evaluation will be slow.")

from models.unet import UNetWithRotationHead
from training.dataset import get_test_loader
from ttt.adapt import run_baseline_inference, run_ttt_inference
from evaluation.metrics import evaluate
print("All imports OK")

In [ ]:
# ── Cell 4: Load checkpoint ───────────────────────────────────────────────────
model = UNetWithRotationHead(n_channels=3, n_classes=1)
ckpt = torch.load('checkpoints/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.to(device)
model.eval()
print(f"Checkpoint loaded (epoch {ckpt.get('epoch', '?')}, "
      f"val_dice={ckpt.get('val_dice', '?'):.4f})")

In [ ]:
# ── Cell 5: Evaluation helper ─────────────────────────────────────────────────
# Best TTT settings from hyperparameter sweep
TTT_LR    = 1e-6
TTT_STEPS = 5
IMG_SIZE  = 512

def run_evaluation(name, image_dir, mask_dir):
    print(f"\n{'='*50}")
    print(f"Dataset: {name}")
    print(f"{'='*50}")

    loader = get_test_loader(image_dir, mask_dir, img_size=IMG_SIZE, batch_size=1)
    n = len(loader)
    print(f"Images: {n}")

    # Baseline
    print("Running baseline inference...")
    base_preds, base_masks = [], []
    for p, m in run_baseline_inference(model, loader, device):
        base_preds.append(p.cpu())
        base_masks.append(m.cpu())
    base_metrics = evaluate(base_preds, base_masks)
    print(f"  Baseline  — Dice: {base_metrics['dice']:.4f}  "
          f"IoU: {base_metrics['iou']:.4f}  "
          f"Sensitivity: {base_metrics['sensitivity']:.4f}  "
          f"Specificity: {base_metrics['specificity']:.4f}  "
          f"AUC: {base_metrics['auc_roc']:.4f}")

    # TTT
    print(f"Running TTT (lr={TTT_LR:.0e}, steps={TTT_STEPS})...")
    ttt_preds, ttt_masks = [], []
    for p, m in run_ttt_inference(model, loader, device, n_steps=TTT_STEPS, lr=TTT_LR):
        ttt_preds.append(p.cpu())
        ttt_masks.append(m.cpu())
    ttt_metrics = evaluate(ttt_preds, ttt_masks)
    delta = ttt_metrics['dice'] - base_metrics['dice']
    print(f"  TTT       — Dice: {ttt_metrics['dice']:.4f}  "
          f"IoU: {ttt_metrics['iou']:.4f}  "
          f"Sensitivity: {ttt_metrics['sensitivity']:.4f}  "
          f"Specificity: {ttt_metrics['specificity']:.4f}  "
          f"AUC: {ttt_metrics['auc_roc']:.4f}")
    print(f"  TTT delta (Dice): {delta:+.4f}")

    return {
        'name': name,
        'n': n,
        'baseline': base_metrics,
        'ttt': ttt_metrics,
        'baseline_preds': base_preds,
        'ttt_preds': ttt_preds,
        'masks': base_masks,
    }

print("Helper ready. TTT settings: lr=1e-6, steps=5")

In [ ]:
# ── Cell 6: Evaluate STARE ────────────────────────────────────────────────────
stare = run_evaluation('STARE', 'data/STARE/images', 'data/STARE/masks')

In [ ]:
# ── Cell 7: Evaluate CHASE ────────────────────────────────────────────────────
chase = run_evaluation('CHASE', 'data/CHASE/images', 'data/CHASE/masks')

In [ ]:
# ── Cell 8: Evaluate HRF ─────────────────────────────────────────────────────
hrf = run_evaluation('HRF', 'data/HRF/images', 'data/HRF/masks')

In [ ]:
# ── Cell 9: Results summary table ────────────────────────────────────────────
results = [stare, chase, hrf]
metrics_keys = ['dice', 'iou', 'sensitivity', 'specificity', 'auc_roc']
metric_labels = ['Dice', 'IoU', 'Sensitivity', 'Specificity', 'AUC-ROC']

print(f"\n{'Dataset':<8} {'Mode':<10} {'Dice':>7} {'IoU':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print("-" * 60)
for r in results:
    for mode, m in [('Baseline', r['baseline']), ('TTT', r['ttt'])]:
        delta = f" ({m['dice']-r['baseline']['dice']:+.4f})" if mode == 'TTT' else ""
        print(f"{r['name']:<8} {mode:<10} "
              f"{m['dice']:>7.4f} {m['iou']:>7.4f} "
              f"{m['sensitivity']:>7.4f} {m['specificity']:>7.4f} "
              f"{m['auc_roc']:>7.4f}{delta}")
    print("-" * 60)

In [ ]:
# ── Cell 10: Bar chart — Dice comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Baseline vs TTT — Dice Score per Dataset', fontsize=14, fontweight='bold')

colors = ['#4C72B0', '#DD8452']
for ax, r in zip(axes, results):
    vals = [r['baseline']['dice'], r['ttt']['dice']]
    bars = ax.bar(['Baseline', 'TTT'], vals, color=colors, width=0.5)
    ax.set_title(r['name'], fontsize=12)
    ax.set_ylabel('Dice Score')
    ax.set_ylim(max(0, min(vals) - 0.05), min(1, max(vals) + 0.05))
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    delta = vals[1] - vals[0]
    ax.set_xlabel(f'TTT delta: {delta:+.4f}', fontsize=10,
                  color='green' if delta >= 0 else 'red')

plt.tight_layout()
plt.savefig('dice_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dice_comparison.png")

In [ ]:
# ── Cell 11: Multi-metric radar/bar chart ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('All Metrics: Baseline vs TTT', fontsize=14, fontweight='bold')

x = np.arange(len(metric_labels))
w = 0.35
for ax, r in zip(axes, results):
    base_vals = [r['baseline'][k] for k in metrics_keys]
    ttt_vals  = [r['ttt'][k] for k in metrics_keys]
    ax.bar(x - w/2, base_vals, w, label='Baseline', color='#4C72B0', alpha=0.85)
    ax.bar(x + w/2, ttt_vals,  w, label='TTT',      color='#DD8452', alpha=0.85)
    ax.set_title(r['name'], fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=20, ha='right')
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('all_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: all_metrics_comparison.png")

In [ ]:
# ── Cell 12: Visual predictions — image | ground truth | baseline | TTT ───────
import torchvision.transforms as T

def show_predictions(result, n_samples=3):
    dataset_name = result['name']
    image_dir = f'data/{dataset_name}/images'
    img_files = sorted(os.listdir(image_dir))[:n_samples]

    fig, axes = plt.subplots(n_samples, 4, figsize=(14, 4 * n_samples))
    fig.suptitle(f'{dataset_name} — Predictions (first {n_samples} images)',
                 fontsize=13, fontweight='bold')

    cols = ['Input Image', 'Ground Truth', 'Baseline Pred', 'TTT Pred']
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=11)

    transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

    for i, fname in enumerate(img_files):
        img = Image.open(os.path.join(image_dir, fname)).convert('RGB')
        img_t = transform(img).permute(1, 2, 0).numpy()

        mask = result['masks'][i].squeeze().numpy()
        base_pred = (result['baseline_preds'][i].squeeze().numpy() > 0.5).astype(float)
        ttt_pred  = (result['ttt_preds'][i].squeeze().numpy() > 0.5).astype(float)

        for ax, im, cmap in zip(axes[i],
                                 [img_t, mask, base_pred, ttt_pred],
                                 [None, 'gray', 'gray', 'gray']):
            ax.imshow(im, cmap=cmap)
            ax.axis('off')

    plt.tight_layout()
    out = f'{dataset_name.lower()}_predictions.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")

show_predictions(stare)
show_predictions(chase)
show_predictions(hrf)

In [ ]:
# ── Cell 13: Save all results to a text file ──────────────────────────────────
with open('eval_results.txt', 'w') as f:
    f.write('Retinal Vessel Segmentation — Evaluation Results\n')
    f.write(f'TTT settings: lr={TTT_LR:.0e}, steps={TTT_STEPS}\n\n')
    for r in results:
        f.write(f"=== {r['name']} ({r['n']} images) ===\n")
        for mode, m in [('Baseline', r['baseline']), ('TTT', r['ttt'])]:
            f.write(f"  {mode}: dice={m['dice']:.4f} iou={m['iou']:.4f} "
                    f"sensitivity={m['sensitivity']:.4f} "
                    f"specificity={m['specificity']:.4f} "
                    f"auc_roc={m['auc_roc']:.4f}\n")
        delta = r['ttt']['dice'] - r['baseline']['dice']
        f.write(f"  TTT delta (Dice): {delta:+.4f}\n\n")

print("Results saved to eval_results.txt")
print("\nDone! All outputs saved in", os.getcwd())

## Part 2: Retrain on Full DRIVE Dataset (40 images)

If you have the DRIVE test images, add them to `data/DRIVE/images` and `data/DRIVE/masks`, then run the cell below to retrain. Skip if you only have 20 images.

In [ ]:
# ── Retrain on all available DRIVE images ────────────────────────────────────
import subprocess, sys
from pathlib import Path

drive_count = len(list(Path('data/DRIVE/images').glob('*')))
print(f"DRIVE images found: {drive_count}")

if drive_count >= 30:
    print("Retraining on full DRIVE dataset...")
    result = subprocess.run(
        [sys.executable, 'training/train.py',
         '--data_root', 'data/DRIVE',
         '--epochs', '60',
         '--batch_size', '4',
         '--lr', '1e-4',
         '--lambda_rot', '0.3',
         '--checkpoint_dir', 'checkpoints_v2'],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode == 0:
        print("\nRetraining complete. New checkpoint saved to checkpoints_v2/")
        CHECKPOINT = 'checkpoints_v2/best_model.pth'
    else:
        print("Error:", result.stderr[-2000:])
else:
    print(f"Only {drive_count} DRIVE images found — need 30+ to retrain.")
    print("Skipping retraining. Using existing checkpoint.")
    CHECKPOINT = 'checkpoints/best_model.pth'


## Part 3: Ablation Study — Baseline vs TTA vs TTT-BN vs TTT-Enc vs TTT-Full

In [ ]:
# ── Ablation study ───────────────────────────────────────────────────────────
import copy
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from training.dataset import get_test_loader
from models.unet import UNetWithRotationHead
from ttt.adapt import test_time_adapt, run_tta_inference
from evaluation.metrics import evaluate

TTT_LR, TTT_STEPS = 1e-6, 5

ABLATION_DATASETS = {
    'STARE': ('data/STARE/images', 'data/STARE/masks'),
    'CHASE': ('data/CHASE/images', 'data/CHASE/masks'),
    'HRF':   ('data/HRF/images',   'data/HRF/masks'),
}

CONDITIONS = [
    ('Baseline',  None),
    ('TTA',       'tta'),
    ('TTT-BN',    'bn'),
    ('TTT-Enc',   'encoder'),
    ('TTT-Full',  'full'),
]

@torch.no_grad()
def run_baseline_abl(mdl, loader):
    mdl.eval(); preds, masks = [], []
    for imgs, msks in loader:
        imgs = imgs.to(device)
        preds.append((torch.sigmoid(mdl(imgs)) > 0.5).float().cpu())
        masks.append(msks.cpu())
    return preds, masks

def run_ttt_abl(mdl, loader, mode):
    preds, masks = [], []
    for imgs, msks in loader:
        imgs = imgs.to(device)
        batch_p = []
        for i in range(imgs.size(0)):
            logits = test_time_adapt(mdl, imgs[i:i+1], n_steps=TTT_STEPS, lr=TTT_LR, adapt_mode=mode)
            batch_p.append((torch.sigmoid(logits) > 0.5).float().cpu())
        preds.append(torch.cat(batch_p)); masks.append(msks.cpu())
    return preds, masks

def run_tta_abl(mdl, loader):
    preds, masks = [], []
    for p, m in run_tta_inference(mdl, loader, device):
        preds.append(p.cpu()); masks.append(m.cpu())
    return preds, masks

ablation_results = {}
for ds_name, (img_dir, mask_dir) in ABLATION_DATASETS.items():
    if not Path(img_dir).exists():
        print(f"Skipping {ds_name}"); continue
    print(f"\n{ds_name}")
    loader = get_test_loader(img_dir, mask_dir, img_size=512, batch_size=1)
    ablation_results[ds_name] = {}
    for cond_name, mode in CONDITIONS:
        print(f"  {cond_name}...", end=' ', flush=True)
        if cond_name == 'Baseline':
            p, m = run_baseline_abl(model, loader)
        elif cond_name == 'TTA':
            p, m = run_tta_abl(model, loader)
        else:
            p, m = run_ttt_abl(model, loader, mode)
        metrics = evaluate(p, m)
        ablation_results[ds_name][cond_name] = metrics
        print(f"Dice={metrics['dice']:.4f}")

# Save ablation results text
with open('ablation_results.txt', 'w') as f:
    f.write(f"{'Dataset':<8} {'Condition':<12} {'Dice':>7} {'IoU':>7} {'Sens':>7} {'Spec':>7}\n")
    f.write("-"*55 + "\n")
    for ds_name, ds_res in ablation_results.items():
        base_dice = ds_res.get('Baseline', {}).get('dice', 0)
        for cond in [c for c, _ in CONDITIONS]:
            if cond not in ds_res: continue
            m = ds_res[cond]; d = m['dice'] - base_dice
            f.write(f"{ds_name:<8} {cond:<12} {m['dice']:>7.4f} {m['iou']:>7.4f} "
                    f"{m['sensitivity']:>7.4f} {m['specificity']:>7.4f}  "
                    f"({'+'if d>=0 else ''}{d:.4f})\n")
print("\nAblation complete. Results saved to ablation_results.txt")

In [ ]:
# ── Ablation results table + chart ───────────────────────────────────────────
cond_names = [c for c, _ in CONDITIONS]
colors_abl = ['#718096', '#63b3ed', '#48bb78', '#f6ad55', '#fc8181']

fig, axes = plt.subplots(1, len(ablation_results), figsize=(5*len(ablation_results), 5),
                         facecolor='#0f1117')
fig.suptitle('Ablation Study — Dice Coefficient by Adaptation Strategy',
             color='white', fontsize=13, fontweight='bold')

for ax, (ds_name, ds_res) in zip(axes if len(ablation_results) > 1 else [axes],
                                  ablation_results.items()):
    dice_vals = [ds_res[c]['dice'] for c in cond_names if c in ds_res]
    bars = ax.bar(range(len(dice_vals)), dice_vals, color=colors_abl, width=0.6)
    ax.set_xticks(range(len(cond_names)))
    ax.set_xticklabels(cond_names, rotation=30, ha='right', color='white', fontsize=9)
    ax.set_title(ds_name, color='white', fontsize=11)
    ax.set_ylabel('Dice', color='white')
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#2d3748')
    ax.set_ylim(0, max(dice_vals) * 1.2 if max(dice_vals) > 0 else 0.1)
    for bar, val in zip(bars, dice_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Saved ablation_study.png")

# Print text table
print(f"\n{'Dataset':<8} {'Cond':<12} {'Dice':>7} {'IoU':>7} {'Sens':>7} {'Spec':>7}")
print("-"*55)
for ds_name, ds_res in ablation_results.items():
    base_dice = ds_res.get('Baseline', {}).get('dice', 0)
    for cond in cond_names:
        if cond not in ds_res: continue
        m = ds_res[cond]
        d = m['dice'] - base_dice
        print(f"{ds_name:<8} {cond:<12} {m['dice']:>7.4f} {m['iou']:>7.4f} "
              f"{m['sensitivity']:>7.4f} {m['specificity']:>7.4f}  "
              f"({'+'if d>=0 else ''}{d:.4f})")

## Part 4: Domain Gap Analysis

In [ ]:
# ── Domain gap analysis ──────────────────────────────────────────────────────
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

DATASETS = {
    'DRIVE': 'data/DRIVE/images',
    'STARE': 'data/STARE/images',
    'CHASE': 'data/CHASE/images',
    'HRF':   'data/HRF/images',
}
COLORS = {'DRIVE': '#63b3ed', 'STARE': '#48bb78', 'CHASE': '#f6ad55', 'HRF': '#fc8181'}

def load_images(folder):
    exts = ['*.png','*.jpg','*.jpeg','*.tif','*.tiff','*.ppm']
    paths = []
    for e in exts: paths.extend(sorted(Path(folder).glob(e)))
    imgs = []
    for p in paths[:20]:
        img = cv2.imread(str(p))
        if img is not None: imgs.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    return imgs

def image_stats(imgs):
    if not imgs: return None
    brightness, contrast, green_mean, green_std = [], [], [], []
    for img in imgs:
        gray = img.mean(axis=2)
        brightness.append(gray.mean()); contrast.append(gray.std())
        green_mean.append(img[:,:,1].mean()); green_std.append(img[:,:,1].std())
    return {'brightness': brightness, 'contrast': contrast,
            'green_mean': green_mean, 'green_std': green_std}

ds_stats = {}
for name, folder in DATASETS.items():
    if not Path(folder).exists(): continue
    imgs = load_images(folder)
    if not imgs: continue
    ds_stats[name] = image_stats(imgs)
    print(f"{name}: {len(imgs)} images | "
          f"brightness={np.mean(ds_stats[name]['brightness']):.1f} | "
          f"contrast={np.mean(ds_stats[name]['contrast']):.1f} | "
          f"green_mean={np.mean(ds_stats[name]['green_mean']):.1f}")

stat_keys = ['brightness', 'contrast', 'green_mean', 'green_std']
stat_labels = ['Brightness (mean)', 'Contrast (std)', 'Green Channel Mean', 'Green Channel Std']

fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor='#0f1117')
fig.suptitle('Domain Gap Analysis', color='white', fontsize=14, fontweight='bold')

for ax, key, label in zip(axes.flat, stat_keys, stat_labels):
    ax.set_facecolor('#1a1d2e')
    ax.set_title(label, color='white', fontsize=10)
    ax.tick_params(colors='white'); ax.yaxis.label.set_color('white')
    for spine in ax.spines.values(): spine.set_edgecolor('#2d3748')
    for ds_name, stats in ds_stats.items():
        vals = stats[key]
        ax.scatter([ds_name]*len(vals), vals, alpha=0.6, color=COLORS.get(ds_name,'white'), s=30)
        ax.plot([ds_name], [np.mean(vals)], marker='D', color=COLORS.get(ds_name,'white'),
                markersize=9, zorder=5)
    ax.set_xticklabels(ax.get_xticklabels(), color='white')

plt.tight_layout()
plt.savefig('domain_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Saved domain_analysis.png")

## Part 5: Intensity Correction (CLAHE) — Testing the Domain-Gap Hypothesis

Part 4 showed that CHASE images are far darker than DRIVE (brightness ~53 vs ~88), and Part 3 showed that TTA *hurts* CHASE while helping STARE/HRF. 

**Hypothesis:** the CHASE failure is driven by the intensity gap, not by vessel structure. If we normalize intensity with CLAHE before inference, the model should improve on CHASE and TTA should stop hurting it.

The cell below builds CLAHE-corrected copies of each test set, re-runs Baseline + TTA on the corrected images, and compares against the originals.


In [ ]:
# ── Build CLAHE-corrected datasets and re-evaluate Baseline + TTA ────────────
import shutil
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from training.dataset import get_test_loader
from ttt.adapt import run_baseline_inference, run_tta_inference, apply_clahe
from evaluation.metrics import evaluate

CORR_DATASETS = {
    'STARE': ('data/STARE/images', 'data/STARE/masks'),
    'CHASE': ('data/CHASE/images', 'data/CHASE/masks'),
    'HRF':   ('data/HRF/images',   'data/HRF/masks'),
}

def make_clahe_dataset(src_img_dir, src_mask_dir, dst_root):
    """Write CLAHE-corrected images to dst_root/images and copy masks unchanged."""
    dst_img  = Path(dst_root) / 'images'
    dst_mask = Path(dst_root) / 'masks'
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_mask.mkdir(parents=True, exist_ok=True)
    for p in sorted(Path(src_img_dir).glob('*')):
        if p.is_file():
            apply_clahe(Image.open(p)).save(dst_img / (p.stem + '.png'))
    for p in sorted(Path(src_mask_dir).glob('*')):
        if p.is_file():
            shutil.copy(p, dst_mask / p.name)
    return str(dst_img), str(dst_mask)

@torch.no_grad()
def eval_baseline(loader):
    preds, masks = [], []
    for p, m in run_baseline_inference(model, loader, device):
        preds.append(p.cpu()); masks.append(m.cpu())
    return evaluate(preds, masks)

def eval_tta(loader):
    preds, masks = [], []
    for p, m in run_tta_inference(model, loader, device):
        preds.append(p.cpu()); masks.append(m.cpu())
    return evaluate(preds, masks)

correction_results = {}
for ds_name, (img_dir, mask_dir) in CORR_DATASETS.items():
    if not Path(img_dir).exists():
        print(f"Skipping {ds_name}"); continue
    print(f"\n{ds_name}")

    # Original images
    orig_loader = get_test_loader(img_dir, mask_dir, img_size=512, batch_size=1)
    base_orig = eval_baseline(orig_loader)
    tta_orig  = eval_tta(get_test_loader(img_dir, mask_dir, img_size=512, batch_size=1))

    # CLAHE-corrected images
    c_img, c_mask = make_clahe_dataset(img_dir, mask_dir, f'data/{ds_name}_clahe')
    base_clahe = eval_baseline(get_test_loader(c_img, c_mask, img_size=512, batch_size=1))
    tta_clahe  = eval_tta(get_test_loader(c_img, c_mask, img_size=512, batch_size=1))

    correction_results[ds_name] = {
        'Baseline (orig)':  base_orig,
        'TTA (orig)':       tta_orig,
        'Baseline (CLAHE)': base_clahe,
        'TTA (CLAHE)':      tta_clahe,
    }
    print(f"  Baseline orig  Dice={base_orig['dice']:.4f}  | TTA orig  Dice={tta_orig['dice']:.4f}")
    print(f"  Baseline CLAHE Dice={base_clahe['dice']:.4f}  | TTA CLAHE Dice={tta_clahe['dice']:.4f}")

# Save text summary
with open('intensity_correction_results.txt', 'w') as f:
    f.write("Intensity Correction (CLAHE) — Dice by condition\n")
    f.write(f"{'Dataset':<8} {'Condition':<18} {'Dice':>7} {'IoU':>7} {'Sens':>7} {'Spec':>7}\n")
    f.write("-" * 58 + "\n")
    for ds_name, conds in correction_results.items():
        for cond, m in conds.items():
            f.write(f"{ds_name:<8} {cond:<18} {m['dice']:>7.4f} {m['iou']:>7.4f} "
                    f"{m['sensitivity']:>7.4f} {m['specificity']:>7.4f}\n")
print("\nSaved intensity_correction_results.txt")


In [ ]:
# ── Intensity-correction chart: original vs CLAHE for Baseline & TTA ─────────
import matplotlib.pyplot as plt
import numpy as np

conds = ['Baseline (orig)', 'TTA (orig)', 'Baseline (CLAHE)', 'TTA (CLAHE)']
cond_colors = ['#718096', '#f6ad55', '#63b3ed', '#48bb78']

n_ds = len(correction_results)
fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 5), facecolor='#0f1117')
if n_ds == 1:
    axes = [axes]
fig.suptitle('Intensity Correction (CLAHE) — Dice by Condition',
             color='white', fontsize=13, fontweight='bold')

for ax, (ds_name, res) in zip(axes, correction_results.items()):
    dice_vals = [res[c]['dice'] for c in conds]
    bars = ax.bar(range(len(conds)), dice_vals, color=cond_colors, width=0.7)
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels(conds, rotation=30, ha='right', color='white', fontsize=8)
    ax.set_title(ds_name, color='white', fontsize=11)
    ax.set_ylabel('Dice', color='white')
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d3748')
    ax.set_ylim(0, max(dice_vals) * 1.25 if max(dice_vals) > 0 else 0.1)
    for bar, val in zip(bars, dice_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('intensity_correction.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Saved intensity_correction.png")

# Text table with CLAHE effect on Dice
print(f"\n{'Dataset':<8} {'Baseline Δ(CLAHE)':>18} {'TTA Δ(CLAHE)':>16}")
print("-" * 44)
for ds_name, res in correction_results.items():
    base_d = res['Baseline (CLAHE)']['dice'] - res['Baseline (orig)']['dice']
    tta_d  = res['TTA (CLAHE)']['dice']      - res['TTA (orig)']['dice']
    print(f"{ds_name:<8} {base_d:>+18.4f} {tta_d:>+16.4f}")


In [ ]:
# ── Final Cell: Download all outputs to your PC ───────────────────────────────
from google.colab import files

output_files = [
    'eval_results.txt',
    'dice_comparison.png',
    'all_metrics_comparison.png',
    'stare_predictions.png',
    'chase_predictions.png',
    'hrf_predictions.png',
    'ablation_results.txt',
    'ablation_study.png',
    'domain_analysis.png',
    'intensity_correction_results.txt',
    'intensity_correction.png',
]

for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
    else:
        print(f"Not found (skipped): {f}")
